# AegisHeaderAudit
**Author:** Sailesh Prasad  
**Project:** Automated Web Scanner for HTTP Security Headers and CORS  

A lightweight defensive auditing tool for websites you own or are explicitly authorized to assess. It checks common response security headers and performs a controlled CORS origin probe. It is not a penetration-testing tool and does not attempt authentication bypass, data extraction, or exploitation.


## Setup
Install dependencies with `pip install -r requirements.txt`. Run this notebook in Jupyter. Set `TARGETS` to URLs you are authorized to test. The sample `https://example.com` is a documentation/example domain; replace it with your own test target where possible.


In [ ]:
import requests
import json
from datetime import datetime, timezone
from urllib.parse import urlparse

SECURITY_HEADERS = {
    'strict-transport-security': 'Encourages HTTPS-only connections (HSTS).',
    'content-security-policy': 'Helps mitigate script injection and content loading risks.',
    'x-content-type-options': 'Prevents MIME-type sniffing when set to nosniff.',
    'x-frame-options': 'Helps control framing/clickjacking.',
    'referrer-policy': 'Controls referrer information sent to other sites.',
    'permissions-policy': 'Controls access to selected browser features.',
}
TARGETS = ['https://example.com']  # Replace with authorized targets only
TIMEOUT = 8
USER_AGENT = 'AegisHeaderAudit/1.0 (authorized defensive security audit)'


In [ ]:
def normalize_url(raw):
    raw = raw.strip()
    if not raw:
        raise ValueError('URL cannot be empty')
    if '://' not in raw:
        raw = 'https://' + raw
    parsed = urlparse(raw)
    if parsed.scheme not in ('http', 'https') or not parsed.hostname:
        raise ValueError(f'Invalid URL: {raw}')
    if parsed.username or parsed.password:
        raise ValueError('Embedded URL credentials are not accepted')
    return raw

def assess_cors(headers):
    origin = headers.get('Access-Control-Allow-Origin')
    credentials = headers.get('Access-Control-Allow-Credentials', '').strip().lower()
    findings = []
    if origin == '*':
        findings.append({'severity': 'INFO', 'finding': 'Wildcard allowed origin',
                         'detail': 'Wildcard origins cannot be used for credentialed browser CORS responses.'})
    if credentials == 'true' and origin == '*':
        findings.append({'severity': 'HIGH', 'finding': 'Wildcard plus credentials',
                         'detail': 'Review this configuration; browsers reject this combination for credentialed requests.'})
    return {'allow_origin': origin, 'allow_credentials': credentials or None,
            'allow_methods': headers.get('Access-Control-Allow-Methods'),
            'allow_headers': headers.get('Access-Control-Allow-Headers'),
            'findings': findings,
            'note': 'Interpret CORS in endpoint context; headers alone do not prove exploitability.'}

def audit_url(raw_url, timeout=TIMEOUT, verify_tls=True):
    url = normalize_url(raw_url)
    result = {'url': url, 'checked_at_utc': datetime.now(timezone.utc).isoformat(),
              'status': None, 'final_url': None, 'https': urlparse(url).scheme == 'https',
              'security_headers': {}, 'missing_security_headers': [], 'cors': {}, 'errors': []}
    session = requests.Session()
    session.headers.update({'User-Agent': USER_AGENT, 'Accept': '*/*'})
    try:
        response = session.get(url, timeout=timeout, allow_redirects=True, verify=verify_tls)
        result['status'] = response.status_code
        result['final_url'] = response.url
        result['https'] = urlparse(response.url).scheme == 'https'
        for name, purpose in SECURITY_HEADERS.items():
            value = response.headers.get(name)
            result['security_headers'][name] = {'present': value is not None, 'value': value, 'purpose': purpose}
            if value is None:
                result['missing_security_headers'].append(name)
        result['cors'] = assess_cors(response.headers)
        # Harmless controlled Origin probe; no credentials or data extraction.
        probe_origin = 'https://aegis-audit.invalid'
        probe = session.get(response.url, headers={'Origin': probe_origin}, timeout=timeout,
                            allow_redirects=True, verify=verify_tls)
        probe_acao = probe.headers.get('Access-Control-Allow-Origin')
        probe_acac = probe.headers.get('Access-Control-Allow-Credentials', '').strip().lower()
        result['cors']['origin_probe'] = {
            'sent_origin': probe_origin, 'returned_allow_origin': probe_acao,
            'returned_allow_credentials': probe_acac or None,
            'reflects_probe_origin': probe_acao == probe_origin}
        if probe_acao == probe_origin and probe_acac == 'true':
            result['cors']['findings'].append({'severity': 'HIGH',
                'finding': 'Probe origin reflected with credentials enabled',
                'detail': 'Review origin validation and whether sensitive responses are exposed.'})
    except requests.RequestException as exc:
        result['errors'].append(str(exc))
    finally:
        session.close()
    return result


In [ ]:
results = []
for target in TARGETS:
    try:
        results.append(audit_url(target))
    except ValueError as exc:
        results.append({'url': target, 'errors': [str(exc)]})

for item in results:
    print('=' * 70)
    print('URL:', item.get('url'))
    print('Status:', item.get('status'), '| HTTPS:', item.get('https'))
    print('Missing headers:', ', '.join(item.get('missing_security_headers', [])) or 'None')
    print('CORS:', json.dumps(item.get('cors', {}), indent=2))
    if item.get('errors'):
        print('Errors:', '; '.join(item['errors']))

with open('aegis_audit_report.json', 'w', encoding='utf-8') as f:
    json.dump({'tool': 'AegisHeaderAudit', 'version': '1.0', 'results': results}, f, indent=2)
print('Saved aegis_audit_report.json')


## Interpretation and limitations
- Missing headers are review items, not automatically vulnerabilities; applications have different requirements.
- CORS findings are indicators, not proof of exploitability. Confirm which endpoints return sensitive data and whether origins are validated correctly.
- The scanner makes ordinary GET requests and one controlled Origin probe per URL. Use only with permission, keep the target list small, and follow the site's testing policy.
- TLS verification is enabled by default. Do not disable it except in a controlled test environment.
